# CircuitVQA — Exploration du pipeline hybride

Ce notebook sert à **voir** ce que fait chaque étage du pipeline sur de vraies images — pas de log wandb ici, juste de l'exploration visuelle.

## Le pipeline, étape par étape

```
Image  ──▶  ①  ──▶  ②  ──▶  ③  ──▶  ④  ──▶  ⑤  ──▶  ⑥
```

| # | Étape | Module | Entrée → Sortie |
|---|---|---|---|
| ① | **Détection** | `Hybrid.Detection.Detector` | image → composants avec bbox (pas encore d'id) |
| ② | **Assignation d'ID** | `Hybrid.Detection.Assign_Ids` | composants bruts → `R1`, `C1`... (numérotés par position de lecture, indépendant des fils — pour que le pipeline reste robuste même si le traçage de fils se trompe) |
| ③ | **Traçage des fils** | `Hybrid.Wires.Tracer` | composants + image → **nets** (quels composants sont électriquement connectés) |
| ④ | **OCR** | `Hybrid.Ocr.Reader` | composants + image → référence lue + valeur lue (ex: `R1`, `4.7kΩ`) |
| ⑤ | **Graphe** | `Graph.Builder` | composants + nets + valeurs → `CircuitGraph` (le pivot du projet) |
| ⑥ | **LLM** | `Llm.Qwen_Client` | netlist (`graph.to_netlist()`) + question → réponse |

**Pourquoi le graphe est le pivot** : tout ce qui précède sert à le construire, tout ce qui suit (LLM, questions/réponses) le consomme. C'est aussi le seul point où l'on peut comparer prédiction et vérité terrain terme à terme (mêmes types Python des deux côtés).

**Deux façons de mesurer** :
- **`ground_truth`** : le graphe vient directement des annotations de `Data_Generation` (parfait par construction) → mesure le **plafond de raisonnement du LLM**, isolé des erreurs d'extraction
- **`pipeline`** : le graphe vient des étapes ①→⑤ réellement exécutées sur l'image → mesure le score **réaliste bout en bout**, incluant toutes les erreurs de détection/fils/OCR

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, 'Src')

import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import networkx as nx
import pandas as pd

from Common.Config import HybridConfig
from Hybrid.Pipeline import HybridPipeline
from Graph.Builder import from_annotation
from Graph.Qa_Generator import generate_qa_pairs
from Llm.Prompting import build_prompt, is_correct
from Llm.Qwen_Client import QwenClient

cfg = HybridConfig.from_yaml('Configs/hybrid.yaml')
dataset_dir = cfg.resolve_path(cfg.data.dataset_dir)

WEIGHTS = 'Runs/hybrid/yolo11n_circuits/weights/best.pt'
pipeline = HybridPipeline.from_config(cfg, weights=WEIGHTS)
print('Pipeline chargé.')

## Choisir un circuit à explorer

In [ ]:
ann_files = sorted((dataset_dir / 'annotations' / 'test').glob('*.json'))
print(f'{len(ann_files)} circuits de test disponibles')

# --- classer par complexité (nb de composants) pour choisir en connaissance de cause ---
by_complexity = []
for f in ann_files:
    a = json.loads(f.read_text(encoding='utf-8'))
    by_complexity.append((len(a['components']), a.get('template', '?'), f))
by_complexity.sort(reverse=True)

print('\n10 circuits les plus complexes du test set :')
for n, tpl, f in by_complexity[:10]:
    print(f'  {n} composants  {tpl:24s}  {f.stem}')

# --- CHOISISSEZ ICI : un cas difficile plutôt que ann_files[0] ---
# soit par index dans le classement (0 = le plus complexe) :
_, _, ann_path = by_complexity[0]
# soit en ciblant un template précis, ex :
# ann_path = next(f for n,t,f in by_complexity if t == 'wheatstone_bridge')

ann = json.loads(ann_path.read_text(encoding='utf-8'))
image_path = dataset_dir / ann['image']
print(f"\nCircuit choisi : {ann_path.stem}  (template: {ann.get('template')}, "
      f"{len(ann['components'])} composants)")

## ① + ② — Image originale, puis détection + IDs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(mpimg.imread(image_path))
axes[0].set_title('① Image originale')
axes[0].axis('off')

result = pipeline.run(image_path)   # exécute ①②③④ d'un coup

img_overlay = cv2.imread(str(image_path))
img_overlay = cv2.cvtColor(img_overlay, cv2.COLOR_BGR2RGB)
for c in result.components:
    x0, y0, x1, y1 = int(c.bbox.x0), int(c.bbox.y0), int(c.bbox.x1), int(c.bbox.y1)
    cv2.rectangle(img_overlay, (x0, y0), (x1, y1), (0, 170, 0), 2)
    cv2.putText(img_overlay, f'{c.id} ({c.confidence:.2f})', (x0, max(15, y0 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 0, 0), 2)
axes[1].imshow(img_overlay)
axes[1].set_title('② Détection + IDs assignés')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'{len(result.components)} composants détectés :')
for c in result.components:
    print(f'  {c.id:6s} {c.cls:20s} confiance={c.confidence:.3f}')

## ③ — Traçage des fils (nets)

In [ ]:
print(f'{len(result.nets)} nets (nœuds électriques) trouvés :\n')
for net in result.nets:
    ids = sorted(net.component_ids())
    print(f"  net {net.net_id}: {' — '.join(ids)}")

## ④ — OCR (référence + valeur lues)

In [ ]:
ocr_rows = []
for comp_id, r in result.ocr.items():
    ocr_rows.append({
        'composant': comp_id,
        'id_lu': r['id_text'],
        'valeur_lue': r['value_text'],
        'texte_brut': r['raw'],
    })
pd.DataFrame(ocr_rows)

## ⑤ — Le graphe (prédit vs vérité terrain, côte à côte)

In [ ]:
CLASS_COLORS = {
    'vsource': '#f39c12', 'battery': '#f39c12',
    'resistor': '#3498db', 'capacitor': '#2ecc71',
    'polarized_capacitor': '#27ae60', 'inductor': '#1abc9c',
    'ground': '#95a5a6', 'diode': '#e67e22', 'zener_diode': '#e74c3c',
    'led': '#e74c3c', 'opamp': '#9b59b6',
    'npn_transistor': '#8e44ad', 'pnp_transistor': '#8e44ad',
}

def draw_graph(ax, graph, title):
    G = nx.Graph()
    for n in graph.nodes:
        label = f'{n.id}\n{n.value}' if n.value else n.id
        G.add_node(n.id, label=label, cls=n.cls)
    for e in graph.edges:
        G.add_edge(e.source, e.target)
    pos = nx.spring_layout(G, seed=42)
    colors = [CLASS_COLORS.get(G.nodes[n]['cls'], '#bdc3c7') for n in G.nodes]
    labels = {n: G.nodes[n]['label'] for n in G.nodes}
    nx.draw(G, pos, ax=ax, node_color=colors, node_size=1600, with_labels=False)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
    ax.set_title(title)

predicted_graph = result.to_graph(circuit_id=ann_path.stem, source_image=str(image_path))
gt_graph = from_annotation(ann_path, circuit_id=ann_path.stem)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
draw_graph(axes[0], predicted_graph, f'Graphe PRÉDIT ({len(predicted_graph.nodes)} nœuds, {len(predicted_graph.edges)} arêtes)')
draw_graph(axes[1], gt_graph, f'Graphe VÉRITÉ TERRAIN ({len(gt_graph.nodes)} nœuds, {len(gt_graph.edges)} arêtes)')
plt.tight_layout()
plt.show()

print('--- netlist prédite (ce que reçoit le LLM) ---')
print(predicted_graph.to_netlist())

## ⑥ — Questions générées + réponses du LLM

In [ ]:
import random
qa_pairs = generate_qa_pairs(gt_graph, rng=random.Random(0))
print(f'{len(qa_pairs)} questions générées automatiquement depuis le graphe (vérité terrain garantie)\n')
pd.DataFrame([p.to_dict() for p in qa_pairs])

In [ ]:
# Charge Qwen2.5-1.5B-Instruct (premier appel : téléchargement ~3 Go)
client = QwenClient()

netlist_for_llm = predicted_graph.to_netlist()   # le pipeline RÉEL, pas la vérité terrain
sample_questions = qa_pairs[:8]   # quelques questions pour explorer, pas tout le banc

rows = []
for qa in sample_questions:
    messages = build_prompt(netlist_for_llm, qa.question)
    raw_answer = client.ask(messages)
    correct = is_correct(raw_answer, qa.answer)
    rows.append({
        'type': qa.question_type,
        'question': qa.question,
        'attendu': qa.answer,
        'réponse_llm': raw_answer,
        'correct': '✅' if correct else '❌',
    })

df = pd.DataFrame(rows)
df

## Explorer un autre circuit

Remontez à la cellule *« Choisir un circuit »*, changez `ann_files[0]` en `ann_files[N]` (N = 0 à 499), et relancez les cellules suivantes.

In [ ]:
import sys
sys.path.insert(0, 'Src')
import torch

from Graph.Builder import from_annotation
from Graph.Tools import build_tools
from Llm.Qwen_Client import QwenClient
from Llm.Tool_Agent import SYSTEM_PROMPT_TOOLS, parse_tool_call

graph = from_annotation('Data/annotations/test/circuit_00000.json', circuit_id='test')
client = QwenClient()
client._ensure_model()
tools = build_tools(graph)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT_TOOLS},
    {"role": "user", "content": "Combien de composants y a-t-il dans ce circuit ?"},
]
text = client._tokenizer.apply_chat_template(
    messages, tools=tools, add_generation_prompt=True, tokenize=False)

model_inputs = client._tokenizer([text], return_tensors="pt").to(client._device)
with torch.no_grad():
    ids = client._model.generate(**model_inputs, max_new_tokens=128)
ids = [o[len(i):] for i, o in zip(model_inputs.input_ids, ids)]
raw = client._tokenizer.batch_decode(ids, skip_special_tokens=True)[0]

# --- résumé court, facile à recopier ---
print("longueur sortie :", len(raw))
print("contient '<tool_call>' :", "<tool_call>" in raw)
print("contient '{' :", "{" in raw)
print("parse_tool_call ->", parse_tool_call(raw))
print("--- 200 premiers caractères ---")
print(raw[:200])

In [ ]:
# CELLULE 1 — diagnostic : quel fichier tourne réellement ?
import inspect, importlib, sys
sys.path.insert(0, 'Src')

import Graph.Tools
importlib.reload(Graph.Tools)   # force le rechargement, contourne le cache

print("fichier utilisé :", Graph.Tools.__file__)

from Graph.Builder import from_annotation
graph = from_annotation('Data/annotations/test/circuit_00000.json', circuit_id='test')
tools = Graph.Tools.build_tools(graph)

# afficher la docstring RÉELLE de la fonction qui plante
fn = [t for t in tools if t.__name__ == 'get_component_class'][0]
print("--- docstring réelle de get_component_class ---")
print(repr(fn.__doc__))

In [ ]:
# CELLULE 2 — tester le parseur directement sur cette fonction
from transformers.utils import get_json_schema
try:
    schema = get_json_schema(fn)
    print("✅ schéma généré :", schema['function']['parameters'])
except Exception as e:
    print("❌", type(e).__name__, ":", e)

In [1]:
import sys
sys.path.insert(0, 'Src')

from Graph.Builder import from_annotation
from Graph.Tools import build_tools
from Llm.Qwen_Client import QwenClient
from Llm.Tool_Agent import ToolAgent

graph = from_annotation('Data/annotations/test/circuit_00065.json', circuit_id='test')
print(f"Circuit : {len(graph.nodes)} composants")

client = QwenClient()
agent = ToolAgent(client, build_tools(graph))

questions = [
    "Combien de composants y a-t-il dans ce circuit ?",
    "Quels sont les composants directement reliés à la masse ?",
    "Est-ce un circuit en série, en parallèle, ou mixte ?",
    "Quels sont les voisins de R1 ?",
    "Quel est le premier composant après la source ?",
]

for q in questions:
    print(f"❓ {q}")
    r = agent.answer(q)
    print(f"   outil : {r['tool_call']}")
    print(f"   résultat : {r['tool_result']}")
    print(f"   réponse : {r['answer']}")
    print()

Circuit : 5 composants
❓ Combien de composants y a-t-il dans ce circuit ?


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

   outil : {'name': 'count_total_components', 'arguments': {}}
   résultat : 5
   réponse : Il y a 5 composants dans ce circuit.

❓ Quels sont les composants directement reliés à la masse ?
   outil : {'name': 'list_components_connected_to_ground', 'arguments': {}}
   résultat : ["Q1", "V1"]
   réponse : Les composants directement reliés à la masse sont Q1 et V1.

❓ Est-ce un circuit en série, en parallèle, ou mixte ?
   outil : {'name': 'get_circuit_topology', 'arguments': {}}
   résultat : mixte
   réponse : Le circuit est mixte.

❓ Quels sont les voisins de R1 ?
   outil : {'name': 'get_neighbours', 'arguments': {'component_id': 'R1'}}
   résultat : []
   réponse : Aucun élément ne correspond à cette requête dans ce circuit.

❓ Quel est le premier composant après la source ?
   outil : {'name': 'get_first_component', 'arguments': {}}
   résultat : RC1
   réponse : Le premier composant après la source est RC1.



In [2]:
# afficher d'abord les vrais composants du circuit
print("Composants disponibles :", [n.id for n in graph.nodes])
print(graph.to_netlist())
print()

questions = [
    "Combien de composants y a-t-il dans ce circuit ?",
    "Quels sont les composants directement reliés à la masse ?",
    "Est-ce un circuit en série, en parallèle, ou mixte ?",
    "Quels sont les voisins de Q1 ?",              # composant qui existe
    "Quelle est la valeur de RC1 ?",               # test des valeurs
    "Quels types de composants y a-t-il et en quelle quantité ?",
    "Y a-t-il des composants non connectés ?",
]

for q in questions:
    print(f"❓ {q}")
    r = agent.answer(q)
    print(f"   outil : {r['tool_call']}")
    print(f"   résultat : {r['tool_result']}")
    print(f"   réponse : {r['answer']}")
    print()

Composants disponibles : ['V1', 'RC1', 'RB1', 'Q1', 'GND1']
# circuit test (electrical)
V1: vsource = 12V
RC1: resistor = 3.3kΩ
RB1: resistor = 8.2kΩ
Q1: npn_transistor
GND1: ground
RC1 -- V1
Q1 -- RB1
GND1 -- Q1
GND1 -- V1
Q1 -- V1
Q1 -- RC1

❓ Combien de composants y a-t-il dans ce circuit ?
   outil : {'name': 'count_total_components', 'arguments': {}}
   résultat : 5
   réponse : Il y a 5 composants dans ce circuit.

❓ Quels sont les composants directement reliés à la masse ?
   outil : {'name': 'list_components_connected_to_ground', 'arguments': {}}
   résultat : ["Q1", "V1"]
   réponse : Les composants directement reliés à la masse sont Q1 et V1.

❓ Est-ce un circuit en série, en parallèle, ou mixte ?
   outil : {'name': 'get_circuit_topology', 'arguments': {}}
   résultat : mixte
   réponse : Le système est mixte.

❓ Quels sont les voisins de Q1 ?
   outil : {'name': 'get_neighbours', 'arguments': {'component_id': 'Q1'}}
   résultat : ["GND1", "RB1", "RC1", "V1"]
   réponse : Le